# Cert Prep — 01: Spark Architecture & Components

**Exam weight: ~17%**

Topics covered:
- Driver vs Executor roles
- Cluster Manager types (Standalone, YARN, Kubernetes)
- Deployment modes: client vs cluster
- Execution hierarchy: Job → Stage → Task
- Lazy evaluation — transformations vs actions
- Fault tolerance: lineage & RDD recomputation
- Garbage collection impact
- SparkContext vs SparkSession


In [1]:
import os, time, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER = 'spark://spark-master:7077'

spark = (
    SparkSession.builder
    .appName('cert-prep')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

# Sample dataset used throughout
data = [
    (1, "Alice",  "Engineering", 95000.0, "2021-03-15", 4),
    (2, "Bob",    "Marketing",   72000.0, "2020-07-01", 3),
    (3, "Carol",  "Engineering", 105000.0,"2019-11-20", 5),
    (4, "Dave",   "Marketing",   68000.0, "2022-01-10", 2),
    (5, "Eve",    "Engineering", 88000.0, "2021-09-05", 4),
    (6, "Frank",  "HR",          61000.0, "2020-04-18", 3),
    (7, "Grace",  "HR",          None,    "2023-02-28", 1),
    (8, "Hank",   "Engineering", 112000.0,"2018-06-12", 5),
    (9, "Iris",   "Marketing",   None,    "2022-11-03", 2),
    (10,"Jack",   "HR",          59000.0, "2023-08-01", 1),
]
schema = StructType([
    StructField("id",         IntegerType(),  False),
    StructField("name",       StringType(),   True),
    StructField("dept",       StringType(),   True),
    StructField("salary",     DoubleType(),   True),
    StructField("hire_date",  StringType(),   True),
    StructField("perf_score", IntegerType(),  True),
])
df = spark.createDataFrame(data, schema)
df.cache().count()
print("Dataset ready")
df.show()

Spark 4.0.2


Dataset ready
+---+-----+-----------+--------+----------+----------+
| id| name|       dept|  salary| hire_date|perf_score|
+---+-----+-----------+--------+----------+----------+
|  1|Alice|Engineering| 95000.0|2021-03-15|         4|
|  2|  Bob|  Marketing| 72000.0|2020-07-01|         3|
|  3|Carol|Engineering|105000.0|2019-11-20|         5|
|  4| Dave|  Marketing| 68000.0|2022-01-10|         2|
|  5|  Eve|Engineering| 88000.0|2021-09-05|         4|
|  6|Frank|         HR| 61000.0|2020-04-18|         3|
|  7|Grace|         HR|    NULL|2023-02-28|         1|
|  8| Hank|Engineering|112000.0|2018-06-12|         5|
|  9| Iris|  Marketing|    NULL|2022-11-03|         2|
| 10| Jack|         HR| 59000.0|2023-08-01|         1|
+---+-----+-----------+--------+----------+----------+



In [2]:
# ── Driver vs Executor ──────────────────────────────────────────────────────
print("=== Spark Application Components ===")
print(f"Driver host     : {spark.sparkContext.master}")
print(f"App name        : {spark.sparkContext.appName}")
print(f"App ID          : {spark.sparkContext.applicationId}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")
print()
print("Driver  — runs main(), holds SparkContext, schedules jobs, collects results")
print("Executor — runs tasks, stores cached data, reports to Driver")
print("Cluster Manager — allocates resources (Standalone / YARN / Kubernetes)")

=== Spark Application Components ===
Driver host     : spark://spark-master:7077
App name        : cert-prep
App ID          : app-20260417184543-0000
Default parallelism: 4

Driver  — runs main(), holds SparkContext, schedules jobs, collects results
Executor — runs tasks, stores cached data, reports to Driver
Cluster Manager — allocates resources (Standalone / YARN / Kubernetes)


In [3]:
# ── Deployment modes ────────────────────────────────────────────────────────
print("=== Deployment Modes ===")
print()
print("CLIENT mode  (default for spark-submit from laptop/notebook):")
print("  - Driver runs on the SUBMITTING machine")
print("  - Output / logs visible in your terminal")
print("  - If submitting machine dies → app dies")
print()
print("CLUSTER mode (production / remote submission):")
print("  - Driver runs inside the cluster (on a worker)")
print("  - Submitting machine can disconnect after submission")
print("  - Better fault tolerance for long-running jobs")
print()
print("LOCAL mode (testing):")
print("  - Everything on one JVM: spark://spark-master:7077 or local[*]")
print("  - local[N] = N threads (tasks), local[*] = all CPU cores")

=== Deployment Modes ===

CLIENT mode  (default for spark-submit from laptop/notebook):
  - Driver runs on the SUBMITTING machine
  - Output / logs visible in your terminal
  - If submitting machine dies → app dies

CLUSTER mode (production / remote submission):
  - Driver runs inside the cluster (on a worker)
  - Submitting machine can disconnect after submission
  - Better fault tolerance for long-running jobs

LOCAL mode (testing):
  - Everything on one JVM: spark://spark-master:7077 or local[*]
  - local[N] = N threads (tasks), local[*] = all CPU cores


In [4]:
# ── Lazy Evaluation: Transformations vs Actions ──────────────────────────────
print("=== Lazy Evaluation ===")
print()
print("Transformations are LAZY — define what to do, nothing executes:")
t = df.filter(F.col("salary") > 80000)       # lazy
t2 = t.select("name", "dept", "salary")      # lazy — no job yet
t3 = t2.withColumn("bonus", F.col("salary") * 0.1)  # lazy
print("  filter → select → withColumn: 0 jobs triggered")
print()

print("Actions TRIGGER execution (submit a job to the cluster):")
# Each of these submits a job
count   = t3.count()          # action → Job submitted
print(f"  .count()   → {count} rows (job executed)")

first   = t3.first()          # action
print(f"  .first()   → {first.name}")

results = t3.collect()        # action — pulls ALL data to driver
print(f"  .collect() → {len(results)} rows pulled to driver")
print()
print("Common actions: count(), collect(), show(), first(), take(n),")
print("                save/write, toPandas(), foreach()")

=== Lazy Evaluation ===

Transformations are LAZY — define what to do, nothing executes:
  filter → select → withColumn: 0 jobs triggered

Actions TRIGGER execution (submit a job to the cluster):
  .count()   → 4 rows (job executed)
  .first()   → Alice
  .collect() → 4 rows pulled to driver

Common actions: count(), collect(), show(), first(), take(n),
                save/write, toPandas(), foreach()


In [5]:
# ── Execution Hierarchy: Job → Stage → Task ──────────────────────────────────
print("=== Execution Hierarchy ===")
print()
print("ACTION  → triggers 1 Job")
print("JOB     → split into Stages at shuffle boundaries (groupBy, join, repartition)")
print("STAGE   → set of tasks that can run in parallel without shuffling")
print("TASK    → unit of work on 1 partition on 1 executor")
print()

# Demonstrate: groupBy forces a shuffle → 2 stages
result = (df
    .filter(F.col("salary").isNotNull())
    .groupBy("dept")
    .agg(F.avg("salary").alias("avg_salary"),
         F.count("*").alias("headcount"))
    .orderBy("dept"))

result.show()
print()
print("Above query plan has:")
print("  Stage 1: filter + partial agg (map-side)")
print("  Stage 2: final agg after shuffle + sort")
result.explain()

=== Execution Hierarchy ===

ACTION  → triggers 1 Job
JOB     → split into Stages at shuffle boundaries (groupBy, join, repartition)
STAGE   → set of tasks that can run in parallel without shuffling
TASK    → unit of work on 1 partition on 1 executor

+-----------+----------+---------+
|       dept|avg_salary|headcount|
+-----------+----------+---------+
|Engineering|  100000.0|        4|
|         HR|   60000.0|        2|
|  Marketing|   70000.0|        2|
+-----------+----------+---------+


Above query plan has:
  Stage 1: filter + partial agg (map-side)
  Stage 2: final agg after shuffle + sort
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [dept#2 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(dept#2 ASC NULLS FIRST, 24), ENSURE_REQUIREMENTS, [plan_id=262]
      +- HashAggregate(keys=[dept#2], functions=[avg(salary#3), count(1)])
         +- Exchange hashpartitioning(dept#2, 24), ENSURE_REQUIREMENTS, [plan_id=259]
            +- HashAggregate(keys=[dep

In [6]:
# ── Fault Tolerance: Lineage ────────────────────────────────────────────────
print("=== Fault Tolerance via RDD Lineage ===")
print()
print("Spark tracks the full transformation lineage (DAG) of every RDD/DataFrame.")
print("If an executor fails mid-task, Spark recomputes ONLY the lost partitions")
print("by replaying the lineage from the last checkpoint or from source data.")
print()
print("This is why Spark does NOT need data replication for fault tolerance.")
print("(Unlike Hadoop HDFS which replicates 3x)")
print()
print("Lineage of our result DataFrame:")
result.explain(mode="extended")

=== Fault Tolerance via RDD Lineage ===

Spark tracks the full transformation lineage (DAG) of every RDD/DataFrame.
If an executor fails mid-task, Spark recomputes ONLY the lost partitions
by replaying the lineage from the last checkpoint or from source data.

This is why Spark does NOT need data replication for fault tolerance.
(Unlike Hadoop HDFS which replicates 3x)

Lineage of our result DataFrame:
== Parsed Logical Plan ==
'Sort ['dept ASC NULLS FIRST], true
+- Aggregate [dept#2], [dept#2, avg(salary#3) AS avg_salary#734, count(1) AS headcount#735L]
   +- Filter isnotnull(salary#3)
      +- LogicalRDD [id#0, name#1, dept#2, salary#3, hire_date#4, perf_score#5], false

== Analyzed Logical Plan ==
dept: string, avg_salary: double, headcount: bigint
Sort [dept#2 ASC NULLS FIRST], true
+- Aggregate [dept#2], [dept#2, avg(salary#3) AS avg_salary#734, count(1) AS headcount#735L]
   +- Filter isnotnull(salary#3)
      +- LogicalRDD [id#0, name#1, dept#2, salary#3, hire_date#4, perf_score

In [7]:
# ── Garbage Collection ───────────────────────────────────────────────────────
print("=== Garbage Collection Impact ===")
print()
print("GC pauses can cause:")
print("  - Executor heartbeat timeouts → spark.executor.heartbeatInterval")
print("  - Task failures if pause > spark.network.timeout (default 120s)")
print()
print("GC tuning strategies:")
print("  1. Use Kryo serializer (already set in spark-defaults.conf)")
print("     spark.serializer = org.apache.spark.serializer.KryoSerializer")
print("  2. Enable off-heap memory to reduce GC pressure:")
print("     spark.memory.offHeap.enabled = true")
print("     spark.memory.offHeap.size = 1g")
print("  3. Use .cache(StorageLevel.DISK_ONLY) for very large DataFrames")
print("  4. Unpersist cached data when no longer needed: df.unpersist()")
print()
print("Key GC metrics to watch in Spark UI → Executors tab:")
print("  - GC Time column (should be < 10% of task duration)")

=== Garbage Collection Impact ===

GC pauses can cause:
  - Executor heartbeat timeouts → spark.executor.heartbeatInterval
  - Task failures if pause > spark.network.timeout (default 120s)

GC tuning strategies:
  1. Use Kryo serializer (already set in spark-defaults.conf)
     spark.serializer = org.apache.spark.serializer.KryoSerializer
  2. Enable off-heap memory to reduce GC pressure:
     spark.memory.offHeap.enabled = true
     spark.memory.offHeap.size = 1g
  3. Use .cache(StorageLevel.DISK_ONLY) for very large DataFrames
  4. Unpersist cached data when no longer needed: df.unpersist()

Key GC metrics to watch in Spark UI → Executors tab:
  - GC Time column (should be < 10% of task duration)
